# Phase 2 experiment campaign (Google Colab)

Runs conditions A/B/C over the 12 pre-registered target modules (`config/libraries.yaml`) with mutation scoring.

**How this survives Colab:** results are written straight into your Google Drive (`MyDrive/thesis_results`) the moment each record exists, so a dying session loses nothing. When a session dies, open this notebook again and `Runtime > Run all`; records already in Drive are skipped and the campaign continues where it stopped. Expect the full campaign to need several sessions (mutation testing dominates the budget).

**Per session you approve two prompts:** the "not authored by Google" warning, and the Google Drive access prompt in cell 1b.

**Done when:** the campaign summary line reports 0 failures and every module has its records in Drive.

In [ ]:
# 1) GPU attached?
!nvidia-smi -L

In [ ]:
# 1b) Google Drive: the durable home for results. Approve the access prompt
# (first run per session); every record lands in MyDrive/thesis_results the
# moment it exists, so a dying session loses nothing and re-runs skip whatever
# Drive already holds.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/thesis_results', exist_ok=True)

In [ ]:
# 2) Clone the repository, or fast-forward an existing clone. The repository is
# public, so this needs no credentials and no Colab secret.
%cd /content
import os
REPO = "https://github.com/youthinkyoucancode/llm-testgen-thesis.git"
if not os.path.isdir("llm-testgen-thesis"):
    !git clone --depth 1 {REPO}
%cd llm-testgen-thesis
!git pull --ff-only {REPO} main


In [ ]:
# 3) Ollama + model (skips whatever already exists)
import shutil, subprocess, time
if shutil.which("ollama") is None:
    !apt-get -qq update && apt-get -qq install -y zstd
    !curl -fsSL https://ollama.com/install.sh | sh
assert shutil.which("ollama"), "Ollama install failed; the installer's ERROR line above says why"
if subprocess.run(["pgrep", "-x", "ollama"], capture_output=True).returncode != 0:
    server = subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
!ollama pull qwen2.5-coder

In [ ]:
# 4) Pinned Python environment
!apt-get -qq install -y python3.12-venv
!python -m venv .venv
import os
assert os.path.exists(".venv/bin/pip"), "venv has no pip; the venv output above says why"
!.venv/bin/pip install -q -r requirements.lock

## Re-run after the context-truncation fix (2026-08-11)

`num_ctx` is now pinned to 32768 in `config/colab.yaml`, and `generate()` raises
if a prompt is ever truncated again. Order for this session:

1. cells 1 to 4 (GPU, Drive, repo pull, Ollama, venv) as usual
2. cell 5 **once** to quarantine the void records
3. cell 6 (pass 1, coverage only) to completion
4. cell 6b to recover the click condition-A records
5. cell 6c (pass 2, mutation) for as many sessions as time allows
6. cell 6d any time to see where things stand, 6e for the statistics

In [ ]:
# 5) Quarantine the void records. SAFE TO RE-RUN: it is a strict one-shot,
# guarded by a sentinel file, because the header tells you to "Run all" when a
# session dies and a second sweep would otherwise move the freshly generated
# B/C records into the void folder and destroy the repaired campaign.
#
# The 2026-08-10 campaign ran with Ollama's default context window, which
# silently truncated every prompt that did not fit, so all 72 B/C records
# measure a fragment of the module rather than the module. They are moved, not
# deleted: they are the evidence for the Ch6 threats paragraph. Condition A
# never calls a model and is left untouched.
import glob, os, shutil
D = '/content/drive/MyDrive/thesis_results'
QUARANTINE_DIR = os.path.join(D, 'voided_20260810')
SENTINEL = os.path.join(QUARANTINE_DIR, '.quarantine_done')

if os.path.exists(SENTINEL):
    n = len(glob.glob(os.path.join(QUARANTINE_DIR, '*.json')))
    print(f'already quarantined ({n} records in voided_20260810/); nothing to do.')
    print('This cell is a one-shot. Records produced since then are safe.')
else:
    os.makedirs(QUARANTINE_DIR, exist_ok=True)
    void = [p for p in glob.glob(os.path.join(D, '*.json'))
            if '_B_s' in os.path.basename(p) or '_C_s' in os.path.basename(p)]
    for p in void:
        shutil.move(p, os.path.join(QUARANTINE_DIR, os.path.basename(p)))
    with open(SENTINEL, 'w') as fh:
        fh.write('quarantined the pre-num_ctx-fix B/C records\n')
    kept = glob.glob(os.path.join(D, '*_A*.json'))
    print(f'quarantined {len(void)} void B/C records -> {QUARANTINE_DIR}')
    print(f'kept {len(kept)} condition-A records (unaffected: A never calls the model)')

In [ ]:
# 6) PASS 1: coverage only, all 12 modules x 3 seeds. Fast (no mutation), so
# this alone produces the complete coverage dataset Ch5 needs. Re-runnable:
# finished records are skipped, failures are logged and retried next pass.
!PYTHONPATH=src .venv/bin/python experiments/run_experiments.py \
    --config config/colab.yaml \
    --conditions BC \
    --skip-mutation \
    --results-dir /content/drive/MyDrive/thesis_results

In [ ]:
# 6b) Condition A repairs. Two independent defects, both found 2026-08-11.
#
# (1) markdown's A records are WRONG, not just missing. One uncollectable module
#     in its suite (a helper class named TestSuite with an __init__) aborted
#     pytest before any test ran, so coverage recorded only import-time
#     statements: inlinepatterns banked 27.8% against a true 99.8%, and
#     blockprocessors 23.9% against 97.7%. Verified locally after the fix.
#     Those two records must be deleted so they rebuild; md_in_html never
#     produced a record at all and now measures 98.0%.
# (2) click's three A records died on the 1800s coverage timeout (its own suite
#     collects 32k+ tests), so they need a bigger budget.
import glob, os, shutil
D = '/content/drive/MyDrive/thesis_results'
BAD_A = os.path.join(D, 'superseded_markdown_A_20260810')
os.makedirs(BAD_A, exist_ok=True)
for stem in ['markdown.inlinepatterns_A', 'markdown.blockprocessors_A']:
    for p in glob.glob(os.path.join(D, stem + '*.json')):
        shutil.move(p, os.path.join(BAD_A, os.path.basename(p)))
        print('superseded:', os.path.basename(p))

!PYTHONPATH=src .venv/bin/python experiments/run_experiments.py \
    --config config/colab.yaml \
    --conditions A --only markdown \
    --coverage-timeout 5400 --mutation-timeout 5400 \
    --results-dir /content/drive/MyDrive/thesis_results

!PYTHONPATH=src .venv/bin/python experiments/run_experiments.py \
    --config config/colab.yaml \
    --conditions A --only click \
    --coverage-timeout 5400 --mutation-timeout 5400 \
    --results-dir /content/drive/MyDrive/thesis_results

In [ ]:
# 6c) PASS 2: mutation. Same command WITHOUT --skip-mutation. The runner's redo
# rule rebuilds any record that has coverage but no mutation score, so this
# fills mutation in on top of pass 1 without redoing generation that already
# landed. Safe to stop and resume across as many sessions as you have time for.
!PYTHONPATH=src .venv/bin/python experiments/run_experiments.py \
    --config config/colab.yaml \
    --mutation-timeout 3600 \
    --results-dir /content/drive/MyDrive/thesis_results

In [ ]:
# 6d) Where does the campaign stand? Run any time.
import glob, json, os
D = '/content/drive/MyDrive/thesis_results'
recs = [json.load(open(p)) for p in glob.glob(os.path.join(D, '*.json'))
        if not p.endswith('_mutation.json')]
bc = [r for r in recs if r['condition'] in ('B', 'C')]
print(f"condition A records : {len([r for r in recs if r['condition']=='A'])} / 12")
print(f"condition B/C       : {len(bc)} / 72")
print(f"  with coverage > 0 : {len([r for r in bc if r['line_percent'] > 0])}")
print(f"  with mutation     : {len([r for r in bc if r.get('mutation_score') is not None])}")

In [ ]:
# 6e) Run the pre-registered analysis (Wilcoxon, Holm-Bonferroni, rank-biserial).
# Reads the per-record JSONs, writes tables into <results-dir>/analysis/.
!PYTHONPATH=src .venv/bin/python experiments/analyze_results.py \
    --results-dir /content/drive/MyDrive/thesis_results

## Reading the results

- `experiments/results/summary.csv`: one flat row per record, the input for the statistics.
- `<module>_A.json` / `<module>_B_s<seed>.json` / `<module>_C_s<seed>.json`: full records including the per-iteration log and the final generated suite.
- `<stem>_mutation.json`: raw mutmut counts and surviving mutant ids per record.
- `failures.log`: every record that could not be produced, with its traceback. A failed record re-runs on the next campaign pass.

When all 12 modules have their records, Phase 2's data collection is done and the statistics scripts take over.